# Servidor Ollama (Qwen) na GPU T4 
Este notebook instala o Ollama, conecta no seu Google Drive (para não ter que baixar os pesos de vários gigabytes toda vez) e expõe a porta de inferência (11434) publicamente usando o **Ngrok** para o seu servidor Flask interagir.

In [ ]:
from google.colab import drive
import os

print("Montando Google Drive...")
drive.mount('/content/drive')

# Prepara uma pasta persistente no GDrive para armazenar os modelos baixados
drive_models_path = '/content/drive/MyDrive/Ollama_Models'
os.makedirs(drive_models_path, exist_ok=True)
print(f"Pasta pronta no Drive: {drive_models_path}")

In [ ]:
!echo "Instalando o Ollama..."
!curl -fsSL https://ollama.com/install.sh | sh

!echo "Criando Symlink da pasta interna pro Google Drive..."
!rm -rf /usr/share/ollama/.ollama
!ln -s /content/drive/MyDrive/Ollama_Models /usr/share/ollama/.ollama
!echo "Pronto! Seus modelos baixados estão sendo guardados no Google Drive (permanentes)."


In [ ]:
!pip install pyngrok

import os
import time
import threading
from pyngrok import ngrok

# RECOMENDADO: Pegue um auth token de ngrok.com/get-started/your-authtoken se bater limitações 
NGROK_TOKEN = ""
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

def start_ollama():
    os.system("OLLAMA_HOST=0.0.0.0 ollama serve")

# Liga o motor do ollama internamente em backgrond
print("Iniciando serviço Ollama local...")
threading.Thread(target=start_ollama, daemon=True).start()
time.sleep(4) 

print("Expondo a porta via ngrok...")
public_url = ngrok.connect(11434).public_url

print("=" * 60)
print(f"-> COPIE ESSA URL ABAIXO NO SEU .env DA SUA MAQUINA!")
print(f"OLLAMA_BASE_URL={public_url}")
print("=" * 60)


In [ ]:
!echo "Iniciando o Download de Qwen2.5:7b (pode demorar na primeira execução se nao estiver salvo no Drive)..."
!ollama pull qwen2.5:7b

!echo "\nTestando o modelo respondendo localmente:"
!ollama run qwen2.5:7b "Fala ai Qwen! Tá pronto pra gerar textos de Memorial? Responda curtinho."